# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. We use the Croissant API to introspect the dataset structure.

**Note:** All entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Enumerate available record sets, fields, and columns by their @id

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets detected in the metadata. Please check dataset schema for record sets.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']} ({rs.get('name','no name')})")
        fields = rs.get('field', [])
        if isinstance(fields, dict):  # Sometimes a single dict instead of list
            fields = [fields]
        for field in fields:
            field_id = field.get('@id') if isinstance(field, dict) else field
            # Get field details if available
            if isinstance(field, dict):
                print(f"  Field: {field_id} - {field.get('name','no name')}")
                # Show columns if present
                if 'column' in field:
                    columns = field['column']
                    if isinstance(columns, dict):
                        columns = [columns]
                    for col in columns:
                        col_id = col.get('@id') if isinstance(col, dict) else col
                        print(f"    Column: {col_id} - {col.get('name','no name') if isinstance(col, dict) else ''}")
            else:
                print(f"  Field: {field_id}")
        print()

## 3. Data Extraction
Load tabular data from each record set into a pandas DataFrame for analysis. Use record set and field `@id` values from the overview step.

Below we extract from all available record sets.

In [ ]:
# Get all record set @id values
record_sets = []
for rs in dataset.record_sets:
    record_sets.append(rs['@id'])

dataframes = {}
for rs_id in record_sets:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for record set {rs_id}")
        else:
            print(f"No records found for record set {rs_id}")
    except Exception as e:
        print(f"Could not load data for record set {rs_id}: {e}")

if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"\nFields/columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No tabular dataframes could be loaded from the dataset.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic filtering, normalization, and grouping operations. 

**Update below field `@id` values accordingly to those printed above.**

In [ ]:
# Select a DataFrame for EDA (use the primary record set loaded above)
if dataframes:
    record_set_id = main_rs_id  # Choose the main loaded record set

    df = dataframes[record_set_id]

    # Replace this with the actual numeric field @id from the extracted DataFrame
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
    else:
        print("No numeric fields detected. Please specify a numeric field @id from the DataFrame.")
        numeric_field = None

    if numeric_field:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Group by a candidate group field (categorical)
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            display(grouped_df.head())
        else:
            print("No categorical group field detected.")
    else:
        print("No numeric field for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field in the dataset using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20, color='dodgerblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No numeric field or DataFrame loaded for visualization.")

## 6. Conclusion
In this notebook, you learned how to use the `mlcroissant` library to:
- Load dataset metadata and inspect its structure using Croissant `@id` references
- Load tabular records into pandas DataFrames
- Conduct basic exploratory analysis: filtering, normalization, grouping
- Visualize data distributions

This workflow provides a reproducible foundation for working with FAIR-compliant datasets described by Croissant schemas. Continue exploring or integrating new analyses as needed!